In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt

In [21]:
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

In [22]:
#Download Data
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, transform=train_transform, download=False)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, transform=test_transform, download=False)

In [23]:
#Define the data loader
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=True)

In [4]:
# Make device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [24]:
# Neural network architecture
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(128 * 4 * 4, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 32x32 -> 16x16
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 16x16 -> 8x8
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # 8x8 -> 4x4

        x = self.dropout(x)

        x = torch.flatten(x, 1)

        # FC layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [25]:
model_0 = NeuralNet().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [26]:
for epoch in range(30):

    epoch_Loss = 0

    for i, batch in enumerate(train_loader):

        optimizer.zero_grad()
        Input, labels = batch
        Input = Input.to(device)
        labels = labels.to(device)
        output = model_0(Input)
        Loss = loss_fn(output, labels)
        Loss.backward()
        optimizer.step()

        epoch_Loss += Loss.item()

    print(f"Epoch:{epoch+1}  Average Batchs Loss: {epoch_Loss/ len(train_loader):.4f}")

Epoch:1  Average Batchs Loss: 1.7882
Epoch:2  Average Batchs Loss: 1.5772
Epoch:3  Average Batchs Loss: 1.4801
Epoch:4  Average Batchs Loss: 1.4099
Epoch:5  Average Batchs Loss: 1.3529
Epoch:6  Average Batchs Loss: 1.3043
Epoch:7  Average Batchs Loss: 1.2667
Epoch:8  Average Batchs Loss: 1.2299
Epoch:9  Average Batchs Loss: 1.1777
Epoch:10  Average Batchs Loss: 1.1278
Epoch:11  Average Batchs Loss: 1.0856
Epoch:12  Average Batchs Loss: 1.0489
Epoch:13  Average Batchs Loss: 1.0262
Epoch:14  Average Batchs Loss: 0.9937
Epoch:15  Average Batchs Loss: 0.9777
Epoch:16  Average Batchs Loss: 0.9603
Epoch:17  Average Batchs Loss: 0.9309
Epoch:18  Average Batchs Loss: 0.9188
Epoch:19  Average Batchs Loss: 0.9025
Epoch:20  Average Batchs Loss: 0.8902
Epoch:21  Average Batchs Loss: 0.8778
Epoch:22  Average Batchs Loss: 0.8566
Epoch:23  Average Batchs Loss: 0.8579
Epoch:24  Average Batchs Loss: 0.8373
Epoch:25  Average Batchs Loss: 0.8331
Epoch:26  Average Batchs Loss: 0.8219
Epoch:27  Average Bat

In [27]:
with torch.no_grad():

    correct = 0
    total = 0
    
    for data in test_loader:
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)
        output = model_0(images)
        total += len(output)
        _, predicted = torch.max(output, 1)
        correct += (predicted == labels).sum().item()

    accuracy = correct/total
    print("Test Accuracy:",accuracy)

Test Accuracy: 0.7022


In [28]:
with torch.no_grad():

    correct = 0
    total = 0
    
    for data in train_loader:
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)
        output = model_0(images)
        total += len(output)
        _, predicted = torch.max(output, 1)
        correct += (predicted == labels).sum().item()

    accuracy = correct/total
    print("Train Accuracy:",accuracy)

Test Accuracy: 0.73076


- Test Accuracy: 0.7022, Train Accuracy: 0.73076
- This is acceptable but we want more
- Now we need to to improve the accuracy of CIFAR10 as much as possible, specifically we want to exceed 92%. To do this, we are going to use CNNs and techniques such as learning rate annealing and data augmentation.

In [2]:
# Data augmentation

train_transform = transforms.Compose([
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.RandomPerspective(distortion_scale=0.5, p=0.5),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

In [3]:
# Make device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
#Download Data
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, transform=train_transform, download=False)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, transform=test_transform, download=False)

C:\Users\body2\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [5]:
#Define the data loader
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=False)

In [6]:
# Define the CNN model
class AdvancedCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(AdvancedCNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32 * 2, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32 * 2, out_channels=64 * 2, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64 * 2, out_channels=64 * 2, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=64 * 2, out_channels=128 * 2, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(in_channels=128 * 2, out_channels=128 * 2, kernel_size=3, padding=1)
        self.conv6 = nn.Conv2d(in_channels=128 * 2, out_channels=128 * 2, kernel_size=3, padding=1)
        self.conv7 = nn.Conv2d(in_channels=128 * 2, out_channels=256 * 2, kernel_size=3, padding=1)
        self.conv8 = nn.Conv2d(in_channels=256 * 2, out_channels=256 * 2, kernel_size=3, padding=1)
        self.conv9 = nn.Conv2d(in_channels=256 * 2, out_channels=256 * 2, kernel_size=3, padding=1)

        self.bn1 = nn.BatchNorm2d(32 * 2)
        self.bn2 = nn.BatchNorm2d(128 * 2)
        self.bn3 = nn.BatchNorm2d(256 * 2)

        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout2d(0.2)

        self.fc1 = nn.Linear(4096 * 2, 4096 * 2)
        self.fc2 = nn.Linear(4096 * 2, 2048 * 2)
        self.fc3 = nn.Linear(2048 * 2, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.maxpool(x)

        x = self.relu(self.bn2(self.conv4(x)))
        x = self.relu(self.conv5(x))
        x = self.relu(self.conv6(x))
        x = self.maxpool(x)
        x = self.dropout(x)

        x = self.relu(self.bn3(self.conv7(x)))
        x = self.relu(self.conv8(x))
        x = self.relu(self.conv9(x))
        x = self.maxpool(x)
        x = self.dropout(x)

        x = torch.flatten(x, start_dim=1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

In [7]:
model_1 = AdvancedCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_1.parameters(), lr=0.01, weight_decay=1e-6, momentum=0.9)
lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=10, min_lr=0.00001)

In [8]:
for epoch in range(50):

    epoch_loss = 0
    train_correct = 0
    train_total = 0
    model_1.train()
    
    for images, labels in train_loader:
        
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        output = model_1(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        _, predicted = torch.max(output, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += len(labels)

    train_accuracy = train_correct / train_total

    test_loss = 0
    test_correct = 0
    test_total = 0
    model_1.eval()

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        output = model_1(images)
        loss = criterion(output, labels)
        test_loss += loss.item()
        
        _, preds = torch.max(output, 1)
        test_correct += (labels == preds).sum().item()
        test_total += len(labels)

    lr_scheduler.step(test_loss)
    test_accuracy = test_correct / test_total

    print(f"Epoch:{epoch+1}\n Loss:{epoch_loss / len(train_loader)}\n Train Accuracy:{train_accuracy}, Test Accuracy:{test_accuracy}")

C:\Users\body2\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\dropout.py:176: UserWarning: dropout2d: Received a 2-D input to dropout2d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout2d exists to provide channel-wise dropout on inputs with 2 spatial dimensions, a channel dimension, and an optional batch dimension (i.e. 3D or 4D inputs).
  return F.dropout2d(input, self.p, self.training, self.inplace)


Epoch:1
 Loss:1.7719722340218318
 Train Accuracy:0.33632, Test Accuracy:0.4762
Epoch:2
 Loss:1.3262215993454727
 Train Accuracy:0.51834, Test Accuracy:0.6256
Epoch:3
 Loss:1.078566360115166
 Train Accuracy:0.61578, Test Accuracy:0.6694
Epoch:4
 Loss:0.9273689616130699
 Train Accuracy:0.67376, Test Accuracy:0.749
Epoch:5
 Loss:0.8142179766299247
 Train Accuracy:0.71408, Test Accuracy:0.7285
Epoch:6
 Loss:0.7383410075747349
 Train Accuracy:0.7433, Test Accuracy:0.7709
Epoch:7
 Loss:0.6696033792871736
 Train Accuracy:0.77018, Test Accuracy:0.8084
Epoch:8
 Loss:0.6114137230854498
 Train Accuracy:0.78864, Test Accuracy:0.8014
Epoch:9
 Loss:0.5737742509738193
 Train Accuracy:0.80216, Test Accuracy:0.8401
Epoch:10
 Loss:0.5285839175594517
 Train Accuracy:0.82, Test Accuracy:0.8343
Epoch:11
 Loss:0.493780152144069
 Train Accuracy:0.8311, Test Accuracy:0.8437
Epoch:12
 Loss:0.4595719128346603
 Train Accuracy:0.84358, Test Accuracy:0.8571
Epoch:13
 Loss:0.43878219550760295
 Train Accuracy:0.8499

In [9]:
torch.save(model_1,"model_1.pth")

In [11]:
torch.save(model_1.state_dict(), "model_1weights.pth")